# Colorectal cancer Visium domain fitting

Fit the configured domain methods from the original public counts, then use the fitted maps for the regional comparisons. This notebook calls the same scientific scripts as the command-line workflow. Select a GPU runtime for the neural methods. BayesSpace uses R and runs on the CPU.

The original publication fits were produced in Colab. This notebook is a new entrypoint for the current code; its full neural/BayesSpace environment has not been tested end to end. To reproduce the specific published fits, use the saved maps with `scripts/reproduce.py` instead.


In [ ]:
from pathlib import Path
import subprocess, sys

REPOSITORY = Path('/content/CRC-Visium-SpatialDomainBenchmark')
if not REPOSITORY.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Zhenghongwei11/CRC-Visium-SpatialDomainBenchmark.git', str(REPOSITORY)], check=True)
if not (REPOSITORY / 'scripts/source_pipeline/fit_source_maps.py').is_file():
    raise FileNotFoundError('Use the repository version containing this notebook and the current source scripts')

from google.colab import drive
drive.mount('/content/drive')
RUN = Path('/content/drive/MyDrive/CRC_Visium/domain_fitting')
RUN.mkdir(parents=True, exist_ok=True)
print('Persistent data and fits:', RUN)


## Scientific dependencies

The fitting environment uses Python 3.11. Installation can take several minutes. R reads the deposited annotations and runs BayesSpace. Keep the same package versions when resuming a fitting run.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'python', 'install', '3.11'], check=True)
SOURCE_ENV = Path('/content/crc-source-environment')
if not SOURCE_ENV.exists():
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.11', str(SOURCE_ENV)], check=True)
SOURCE_PYTHON = SOURCE_ENV / 'bin/python'
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(SOURCE_PYTHON), '-r', str(REPOSITORY / 'environments/neural-python311.txt')], check=True)

wheel_url = subprocess.check_output([str(SOURCE_PYTHON), '-c', "import torch; v=torch.__version__.split('+')[0]; c='cpu' if torch.version.cuda is None else 'cu'+torch.version.cuda.replace('.',''); print(f'https://data.pyg.org/whl/torch-{v}+{c}.html')"], text=True).strip()
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(SOURCE_PYTHON), '--only-binary', ':all:', '-r', str(REPOSITORY / 'environments/pyg-extensions.txt'), '--find-links', wheel_url], check=True)

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'r-base', 'r-base-dev', 'libhdf5-dev', 'libcurl4-openssl-dev', 'libssl-dev', 'libxml2-dev'], check=True)
subprocess.run(['Rscript', '-e', 'options(repos=c(CRAN="https://cloud.r-project.org")); install.packages(c("BiocManager","mclust","hdf5r")); BiocManager::install(c("SingleCellExperiment","BayesSpace"),ask=FALSE,update=FALSE)'], check=True)


## Original public inputs

Downloaded archives and extracted original members remain in Drive. The source configuration records their identities. A repeated invocation resumes acquisition rather than replacing the data.


In [ ]:
INPUTS = RUN / 'original_inputs'
command = [str(SOURCE_PYTHON), str(REPOSITORY / 'scripts/source_pipeline/source_inputs.py'), '--cache', str(RUN / 'raw_cache'), '--output', str(INPUTS), '--workers', '2']
if INPUTS.exists():
    command.append('--resume')
subprocess.run(command, check=True)


## Domain fits

The comparison includes 22 sections, six map families for Valdeolivas and seven for GSE294385. BayesSpace is included only for GSE294385. Each completed section is saved before the next section begins. On reconnect, run the setup cells and this cell again; completed sections are checked and reused. An interrupted section is fitted again.


In [ ]:
FITS = RUN / 'source_fits'
command = [str(SOURCE_PYTHON), str(REPOSITORY / 'scripts/source_pipeline/fit_source_maps.py'), '--inputs', str(INPUTS), '--output', str(FITS), '--methods', 'all']
if FITS.exists():
    command.append('--resume')
subprocess.run(command, check=True)


## Generate regional results and figures

This step uses Python 3.13.7, the environment of the current regional analyses. It calculates regional measurements from the new fits, rather than substituting the submitted tables. Its results can differ from the publication when fitting software or hardware changes.

Alternatively, download `source_fits/` and `original_inputs/counts/` from Drive and run the command in the README on your computer.


In [ ]:
subprocess.run([sys.executable, '-m', 'uv', 'python', 'install', '3.13.7'], check=True)
ANALYSIS_ENV = Path('/content/crc-analysis-environment')
if not ANALYSIS_ENV.exists():
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.13.7', str(ANALYSIS_ENV)], check=True)
ANALYSIS_PYTHON = ANALYSIS_ENV / 'bin/python'
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(ANALYSIS_PYTHON), '-r', str(REPOSITORY / 'requirements.txt')], check=True)
RESULTS = RUN / 'regional_results'
command = [str(ANALYSIS_PYTHON), str(REPOSITORY / 'scripts/reproduce.py'), '--source-fits', str(FITS), '--counts', str(INPUTS / 'counts'), '--output', str(RESULTS), '--workers', '2']
if RESULTS.exists():
    command.append('--resume')
subprocess.run(command, check=True)
print('Tables:', RESULTS / 'tables')
print('Figures:', RESULTS / 'figures')
